# MCS Case Study 1 — AMP dataset builder (from pivot table)

This notebook:
1. Loads the provided pivot table (`pivoted_sequences.csv`).
2. Builds a **raw AMP case-study dataset** using the `Antimicrobial` column as target.
3. Subsamples up to **10,000** sequences (stratified by label) for practical experimentation.
4. Exports:
   - `amp_raw_10k.csv` (raw subset)
   - `splits/` (train/val/test indices + split csvs)
   - `dataset.yaml` (MCS-style dataset contract / manifest)

> **Reproducibility**: sampling + splits are controlled via a single `SEED`.

In [22]:
# --- Config ---
from pathlib import Path

In [23]:
SRC_CSV = Path(r"../../raw_data_use_cases/raw_data_peptides.csv")
OUT_DIR = Path(r"../../demonstrative_applications/case_demo_01/dataset/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR_SCHEMAS = Path(r"../../demonstrative_applications/case_demo_01/schemas/")
OUT_DIR_SCHEMAS.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Antimicrobial"
SEQ_COL = "sequence"

MAX_N = 10_000          # practical cap requested
SEED = 42               # change if you want a different deterministic sample
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9
print("Output folder:", OUT_DIR)

Output folder: ../../demonstrative_applications/case_demo_01/dataset


In [24]:
import pandas as pd

df = pd.read_csv(SRC_CSV)
print("Shape:", df.shape)
print("Columns:", len(df.columns))
df.head()

Shape: (86477, 214)
Columns: 214


,sequence,Anti_HIV,Therapeutic,Anti_nematode,Anti_coronaviridae,Anti_feline_coronavirus,Enzyme_inhibitor,Anti_west_nile_virus,Anti_junin_virus,Anti_iridoviridae,...,Anti_human_coronavirus,Anti_human_parainfluenza_virus,Anti_puumala_virus,Anti_amnesic,Cytokine,Anti_hendra_virus,Potentiator,Neuropeptide,Anti_white_spot_syndrome_virus,Anti_herpesviridae
0,MFTLKKTLLLLFFIGTISLSLCEQERDADEDEGEALEEVKRGLWDT...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,QMIVIELGTNPLK,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,PTSRGNNRRPQTRGMVEECCFRSCDLNLLEQYCAKPAKSERDVSAT...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,RCICTLGVC,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,MNFKYIVAVSFLIASGYARSVRNDEQSLSQRDVLEEESPREIRGIG...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
# --- Sanity checks ---
assert SEQ_COL in df.columns, f"Missing {SEQ_COL}"
assert TARGET_COL in df.columns, f"Missing {TARGET_COL}"

# Keep only the relevant columns for this case-study
raw = df[[SEQ_COL, TARGET_COL]].copy()

# Basic cleanup
raw[SEQ_COL] = raw[SEQ_COL].astype(str).str.strip()
raw = raw[raw[SEQ_COL].str.len() > 0]

# Ensure target is binary 0/1
raw[TARGET_COL] = pd.to_numeric(raw[TARGET_COL], errors="coerce").astype("Int64")
raw = raw.dropna(subset=[TARGET_COL]).copy()
raw[TARGET_COL] = raw[TARGET_COL].astype(int)
assert set(raw[TARGET_COL].unique()).issubset({0, 1})

# Remove exact duplicate sequences (keeping first occurrence)
n_before = len(raw)
raw = raw.drop_duplicates(subset=[SEQ_COL], keep="first").reset_index(drop=True)
print(f"Removed {n_before - len(raw)} duplicate sequences")

# Quick stats
raw["length"] = raw[SEQ_COL].str.len()
display(raw[TARGET_COL].value_counts().rename("count"))
raw["length"].describe()

Removed 0 duplicate sequences


Antimicrobial
0    52067
1    34410
Name: count, dtype: int64

count    86477.000000
mean        34.889786
std         31.461140
min          2.000000
25%         14.000000
50%         23.000000
75%         45.000000
max        150.000000
Name: length, dtype: float64

In [26]:
# --- Subsample up to MAX_N (stratified) ---
import numpy as np

rng = np.random.default_rng(SEED)

if len(raw) <= MAX_N:
    subset = raw.drop(columns=["length"]).copy()
    print("No subsampling needed; using full dataset:", len(subset))
else:
    # Stratified sampling to preserve label proportions
    subset_parts = []
    for y, part in raw.groupby(TARGET_COL, sort=False):
        frac = MAX_N / len(raw)
        k = max(1, int(round(frac * len(part))))
        idx = rng.choice(part.index.to_numpy(), size=min(k, len(part)), replace=False)
        subset_parts.append(raw.loc[idx])

    subset = pd.concat(subset_parts, axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    # If rounding overshoots, trim deterministically
    if len(subset) > MAX_N:
        subset = subset.iloc[:MAX_N].copy()

    subset = subset.drop(columns=["length"], errors="ignore")
    print("Subsampled:", len(subset))

display(subset[TARGET_COL].value_counts().rename("count"))
subset.head()

Subsampled: 10000


Antimicrobial
0    6021
1    3979
Name: count, dtype: int64

,sequence,Antimicrobial
0,MASHPNLRRSENSFALSDTDLLPGVENFEINSVEEKLLEELGSYDE...,0
1,MMKRLIVLVLLASTLLTGCNTARGFGEDIKHLGNSISRAAS,0
2,MVTIRLARHGAKKRPFYQVVVADSRNARNGRFIERVGFFNPIASEK...,1
3,SEDRSTGNSLKDSSSFSPARY,0
4,RNIEPLNNDAMASLLSVANFKHVPAVS,0


In [27]:
# --- Export raw subset ---
raw_csv = OUT_DIR / "amp_raw_10k.csv"
subset.to_csv(raw_csv, index=False)

# Compute sha256 for verifiability
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

raw_sha256 = sha256_file(raw_csv)
print("Wrote:", raw_csv)
print("sha256:", raw_sha256)

Wrote: ../../demonstrative_applications/case_demo_01/dataset/amp_raw_10k.csv
sha256: 1b509a0a745a14602c1d2d6738327c5995d79ac6ceb26c2486caf69edccee2b0


In [28]:
# --- Generate MCS-style dataset YAML (manifest/contract) ---
import yaml
from datetime import datetime, timezone

manifest = {
    "mcs_version": "0.1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "id": "amp_case_study_v1",
        "name": "Antimicrobial Peptide (AMP) case-study subset",
        "description": (
            "Binary classification dataset derived from a pivoted peptide table. "
            "The target label is 'Antimicrobial' (0/1). A stratified random subset "
            f"of up to {MAX_N} sequences is selected for practical experimentation."
        ),
        "domain": "peptides",
        "task": "binary_classification",
        "features": [{"name": SEQ_COL, "type": "string"}],
        "target": {"name": TARGET_COL, "type": "int", "labels": {0: "non_AMP", 1: "AMP"}},
        "source": {
            "type": "local_file",
            "path": str(SRC_CSV),
        },
        "sampling": {
            "method": "stratified_random",
            "max_n": int(MAX_N),
            "seed": int(SEED),
            "notes": "Sampling performed after removing empty sequences and exact duplicate sequences."
        },
        "artifacts": {
            "raw_subset_csv": "amp_raw_10k.csv",
            "raw_subset_sha256": raw_sha256,
        },
        "statistics": {
            "n_total": int(len(subset)),
            "label_counts": subset[TARGET_COL].value_counts().to_dict(),
            "seq_length": {
                "min": int(subset[SEQ_COL].str.len().min()),
                "max": int(subset[SEQ_COL].str.len().max()),
                "mean": float(subset[SEQ_COL].str.len().mean()),
                "median": float(subset[SEQ_COL].str.len().median()),
            },
        },
        "license": "MIT",
        "citation": "TBD",
        "contact": {
            "owner": "KrenAI Lab",
            "email": "krenai@umag.cl"
        }
    }
}

yaml_path = OUT_DIR_SCHEMAS / "dataset.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(manifest, f, sort_keys=False, allow_unicode=True)

print("Wrote:", yaml_path)
print(yaml_path.read_text()[:800])

Wrote: ../../demonstrative_applications/case_demo_01/schemas/dataset.yaml
mcs_version: '0.1'
created_utc: '2026-01-25T17:06:59.513657+00:00'
dataset:
  id: amp_case_study_v1
  name: Antimicrobial Peptide (AMP) case-study subset
  description: Binary classification dataset derived from a pivoted peptide table.
    The target label is 'Antimicrobial' (0/1). A stratified random subset of up to
    10000 sequences is selected for practical experimentation.
  domain: peptides
  task: binary_classification
  features:
  - name: sequence
    type: string
  target:
    name: Antimicrobial
    type: int
    labels:
      0: non_AMP
      1: AMP
  source:
    type: local_file
    path: ../../raw_data_use_cases/raw_data_peptides.csv
  sampling:
    method: stratified_random
    max_n: 10000
    seed: 42
    notes: Sampling performed after removing empty sequences and exact
